# Redrob Hackathon — EDA & Scoring Exploration
Analyse sample candidates, inspect score distributions, tune weights.

In [ ]:
import sys, json
sys.path.insert(0, '..')
import pandas as pd

# Load sample candidates
with open('../data/sample_candidates.json') as f:
    candidates = json.load(f)
print(f'Loaded {len(candidates)} sample candidates')

In [ ]:
from src.preprocess import preprocess_candidate
from src.ranking_engine import score_candidate, rank_candidates
from src.explainability import generate_reasoning

# Preprocess and score all samples
scored = []
for raw in candidates:
    cleaned = preprocess_candidate(raw)
    if cleaned:
        s = score_candidate(cleaned)
        s['reasoning'] = generate_reasoning(s)
        scored.append(s)

ranked = rank_candidates(scored)
print(f'Scored {len(ranked)} candidates')

In [ ]:
# Build analysis dataframe
rows = []
for s in ranked:
    c = s['_candidate']
    p = c.get('profile', {})
    rows.append({
        'rank': s['rank'],
        'candidate_id': s['candidate_id'],
        'title': p.get('current_title',''),
        'yoe': p.get('years_of_experience', 0),
        'location': p.get('location', ''),
        'final_score': s['final_score'],
        'skill_score': s['skill_score'],
        'exp_score': s['exp_score'],
        'career_score': s['career_score'],
        'behavior_score': s['behavior_score'],
        'edu_score': s['edu_score'],
        'loc_score': s['loc_score'],
        'prod_depth_score': s['prod_depth_score'],
        'trap_penalty': s['trap_penalty'],
        'is_honeypot': s['is_honeypot'],
        'trap_flags': '; '.join(s.get('trap_flags', [])),
        'reasoning': s['reasoning'],
    })
df = pd.DataFrame(rows)
df.head(10)

In [ ]:
# Score distribution
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
cols = ['final_score','skill_score','exp_score','career_score',
        'behavior_score','edu_score','loc_score','prod_depth_score']
for ax, col in zip(axes.flat, cols):
    ax.hist(df[col], bins=20, color='steelblue', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel('Score')
plt.tight_layout()
plt.savefig('../outputs/score_distributions.png')
plt.show()

In [ ]:
# Top 10 candidates
pd.set_option('display.max_colwidth', 80)
df[['rank','candidate_id','title','yoe','location','final_score','reasoning']].head(10)

In [ ]:
# Trap detection analysis
print(f'Honeypots detected: {df.is_honeypot.sum()}')
print(f'Candidates with any flag: {(df.trap_penalty > 0).sum()}')
print()
print('Flagged candidates:')
df[df.trap_penalty > 0][['candidate_id','title','final_score','trap_penalty','trap_flags']].head(15)

In [ ]:
# Skill group coverage heatmap
import numpy as np

group_data = []
for s in ranked[:20]:
    groups = s.get('skill_debug', {}).get('must_have_groups', {})
    group_data.append(groups)

gdf = pd.DataFrame(group_data).fillna(0)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(gdf.values, cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(gdf.columns)))
ax.set_xticklabels(gdf.columns, rotation=30, ha='right')
ax.set_yticks(range(len(gdf)))
ax.set_yticklabels([f'Rank {i+1}' for i in range(len(gdf))])
ax.set_title('Must-have skill group coverage (top 20)')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation between components and final score
score_cols = ['skill_score','exp_score','career_score','behavior_score',
              'edu_score','loc_score','prod_depth_score']
correlations = df[score_cols + ['final_score']].corr()['final_score'].drop('final_score')
correlations.sort_values(ascending=False).plot(kind='bar', color='steelblue')
plt.title('Correlation with final_score')
plt.ylabel('Pearson r')
plt.tight_layout()
plt.show()

## Advanced: Add semantic score
If you want to add sentence-transformer embeddings as a bonus (pre-compute offline),
run this cell ONCE to generate embeddings, then use them in ranking_engine.py.

In [ ]:
# OPTIONAL — requires: pip install sentence-transformers
# from sentence_transformers import SentenceTransformer
# import numpy as np
#
# JD_TEXT = """
# Senior AI Engineer specialising in embedding systems, vector search,
# retrieval-augmented generation, semantic search, Pinecone, Qdrant, FAISS,
# sentence-transformers, Python, LLM fine-tuning, learning-to-rank.
# """
#
# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
# jd_vec = model.encode([JD_TEXT], normalize_embeddings=True)[0]
#
# def get_candidate_text(c):
#     p = c.get('profile', {})
#     return (p.get('headline','') + ' ' + p.get('summary','') + ' '
#             + ' '.join(s.get('name','') for s in c.get('skills',[])))
#
# texts = [get_candidate_text(raw) for raw in candidates]
# vecs = model.encode(texts, batch_size=64, normalize_embeddings=True)
# sims = vecs @ jd_vec   # cosine similarities
# print('Semantic scores computed:', sims.shape)